## Qwen/Qwen2.5-14B-Instruct-AWQ

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/root/bukan-skripsi/codes/test-translate/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# We pull a version that has already been compressed to 4-bit by the community
model_id = "Qwen/Qwen2.5-14B-Instruct-AWQ"

print("Downloading pre-quantized blocks directly to GPU...")

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Stream the 4-bit files straight into your hardware
# This will bypass your 12GB system RAM limitation and fit easily into your 16GB VRAM
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda" # Automatically targets your GPU
)

print("Model successfully loaded onto your GPU! Ready to chat.")

W0816 13:41:09.718000 1141047 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0816 13:41:09.833000 1141047 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.3.2+ab5a1e7
Transformers : 5.15.0
Torch        : 2.13.0+cu130
Triton       : 3.7.1
INFO  Kernel: Auto-selection: adding candidate `AwqMarlinLinear`                

Loading weights:   0%|          | 0/1251 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# 3. Use the model immediately
messages = [
    {"role": "system", "content": "Translate the following English social-media text into natural Indonesian. Preserve the original meaning, tone, ambiguity, slang, exaggeration, sarcasm, and pragmatic cues as closely as possible. Do not explain the text, resolve ambiguity, infer unstated meaning, or add information. Return only the translated text."},
    {"role": "user", "content": "What a successful toast, it looks so delicious!"}
]

# Format prompt using Qwen's specific template structure
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate response
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

# Extract only the newly generated text fragments
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print("\n--- Model Response ---")
print(response[0])


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-14B-Instruct-AWQ"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


## Deepseek-r1:14b

In [1]:
import ollama

In [ ]:
model_name = "deepseek-r1:14b"

stream = ollama.chat(
    model=model_name,
    messages=[
    {"role": "system", "content": "Translate the following English social-media text into natural Indonesian. Preserve the original meaning, tone, ambiguity, slang, exaggeration, sarcasm, and pragmatic cues as closely as possible. Do not explain the text, resolve ambiguity, infer unstated meaning, or add information. Return only the translated text."},
    {"role": "user", "content": "What a successful toast, it looks so delicious!"},
    {"role": "user", "content": "You must be fun at parties"},
    {"role": "user", "content": "What a great idea, even the undead will be dreadful"}
    ],
    stream=True,
    options={'temperature': 0}
)

# Print tokens as they arrive from the local server
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)




Apa itu toast yang sukses, terlihat sangatlezat!  
Kamu pasti menyenangkan di pesta.  
Apa ide yang bagus! bahkan zombie pun akan merasa takut.